# C3-gradient-descent — Practice p14 — Solution

Full-batch descent reaches the same final loss, about $0.0949943$, at both learning rates because its deterministic gradients settle at the minimum. Mini-batch descent has floors about $0.09732$ for $\eta=0.1$ and $0.10811$ for $\eta=0.3$, so its final quality depends more strongly on the step size. Random batch gradients keep kicking the iterate around the minimum, and the larger learning rate turns those noisy gradients into larger displacements.

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)
X = rng.normal(0, 1, (300, 3))
w_true = np.array([0.5, -1.5, 2.5])
b_true = 1.0
y = (X * w_true).sum(axis=1) + b_true + rng.normal(0, 0.3, 300)
rng_batch = np.random.default_rng(SEED)
batch_indices = rng_batch.integers(0, 300, size=(300, 10))

def mse_loss(X, y, w, b):
    return (((X * w).sum(axis=1) + b - y) ** 2).mean()

def mse_gradients(X, y, w, b):
    residuals = (X * w).sum(axis=1) + b - y
    return 2 * (residuals[:, None] * X).mean(axis=0), 2 * residuals.mean()

def run_full(eta):
    w = np.zeros(3)
    b = 0.0
    losses = np.empty(300)
    for step in range(300):
        grad_w, grad_b = mse_gradients(X, y, w, b)
        w = w - eta * grad_w
        b = b - eta * grad_b
        losses[step] = mse_loss(X, y, w, b)
    return losses

def run_sgd(eta):
    w = np.zeros(3)
    b = 0.0
    losses = np.empty(300)
    for step in range(300):
        idx = batch_indices[step]
        grad_w, grad_b = mse_gradients(X[idx], y[idx], w, b)
        w = w - eta * grad_w
        b = b - eta * grad_b
        losses[step] = mse_loss(X, y, w, b)
    return losses

loss_full = run_full(0.1)
full_final = loss_full[-1]
loss_sgd = run_sgd(0.1)
sgd_final = loss_sgd[-1]
sgd_floor = loss_sgd[-50:].mean()
examples_full = 300 * 300
examples_sgd = 300 * 10
cost_ratio = examples_full / examples_sgd
loss_sgd_big = run_sgd(0.3)
sgd_floor_big = loss_sgd_big[-50:].mean()
loss_full_big = run_full(0.3)
full_final_big = loss_full_big[-1]
full_final, sgd_final, sgd_floor, examples_full, examples_sgd, cost_ratio, sgd_floor_big, full_final_big

### Answer check

In [ ]:
assert loss_full.shape == (300,) and loss_sgd.shape == (300,)
assert np.isclose(full_final, 0.094994308483, atol=1e-12)
assert np.isclose(sgd_final, 0.096315328953, atol=1e-12)
assert np.isclose(sgd_floor, 0.097324598107, atol=1e-12)
assert examples_full == 90000 and examples_sgd == 3000 and np.isclose(cost_ratio, 30.0)
assert np.isclose(sgd_floor_big, 0.108113287679, atol=1e-12)
assert np.isclose(full_final_big, 0.094994308483, atol=1e-12)
assert sgd_floor_big > sgd_floor